<a href="https://colab.research.google.com/github/doanxuantuong/PlantLeafDiseaseClassification/blob/tuong/Plant_Leaf_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q timm scikit-learn seaborn pandas matplotlib pillow tqdm

In [19]:
#Import, cấu hình và tái lập thực nghiệm ghi hết ở cell này
import os, random, time, json, hashlib, shutil, warnings
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_style("whitegrid")

#tái lập thực nghiệm
SEED = 42
def seed_everything(s=SEED):
    random.seed(s);
    np.random.seed(s);
    torch.manual_seed(s);
    torch.cuda.manual_seed_all(s)
seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("DÙng t4 GPT")

CFG = dict(
    IMG_SIZE=224,
    BATCH=64,
    EPOCHS=6,
    FREEZE_EPOCHS=1,      # Nếu ep == 0 thì đóng băng, đến ep == 1 thì mở toàn bộ
    PATIENCE=3,           # đếm số epoch ko cải thiện, đủ 3 thì dừng
    WD=0.0,
    NUM_WORKERS=2,
    MAX_PER_CLASS=None,   # đặt 100 để chạy thử nhanh; None = dùng toàn bộ dữ liệu
    OUTLIER_RATE=0.01,    # tỉ lệ ảnh bị đánh dấu bất thường trong mỗi lớp
    WORK_DIR="/content/work",
    DRIVE_OUT="/content/drive/MyDrive/N4k1/DataMining/datamining_plant_out",
)


# làm đẹp tên nhãn
def pretty(label):
    return label.replace("___", " - ").replace("_", " ")

# show nhiều ảnh
def show_grid(paths, titles, ncols=4, size=3):
    n = len(paths); nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, p, t in zip(axes, paths, titles):
        ax.imshow(Image.open(p).convert("RGB")); ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

#tiền xử lí ảnh để đánh giá
def make_eval_tf(mean, std, size=None):
    size = size or CFG["IMG_SIZE"]
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(mean, std)])

# tự xây dựng dataset cho ảnh
# class ImgDS(Dataset):
#     def __init__(self, paths, labels, tf):
#         self.paths = list(paths); self.labels = list(labels); self.tf = tf
#     def __len__(self):
#         return len(self.paths)
#     def __getitem__(self, i):
#         img = Image.open(self.paths[i]).convert("RGB")
#         return self.tf(img), int(self.labels[i])

GPU: Tesla T4


In [7]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Không ở môi trường Colab hoặc Drive đã được mount:", e)

SRC_DIR = "/content/drive/MyDrive/N4k1/DataMining/PlantVillage_Dataset"
OUT = Path(CFG["DRIVE_OUT"]);
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path(CFG["WORK_DIR"]) / "raw";
RAW.mkdir(parents=True, exist_ok=True)

def extract_all(src, dst):
    for f in sorted(Path(src).iterdir()):
        print(f"- {f.name}  ({f.stat().st_size / 1e6:.1f} MB)" if f.is_file() else f"- {f.name}/ (thư mục)")
        if not f.is_file():
            continue
        out = dst / f.stem
        if out.exists():
            print("   đã giải nén trước đó, bỏ qua"); continue
        try:
            shutil.unpack_archive(str(f), str(out)); print("   giải nén ->", out)
        except (shutil.ReadError, ValueError):
            print("   không giải nén được (.rar/.7z cần: !apt-get -qq install unrar p7zip-full)")

extract_all(SRC_DIR, RAW)
for z in list(RAW.rglob("*.zip")):            # file zip lồng trong zip
    out = z.with_suffix("")
    if not out.exists():
        shutil.unpack_archive(str(z), str(out))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
- Plant_leaf_diseases_dataset_with_augmentation.zip  (949.3 MB)
   đã giải nén trước đó, bỏ qua
- Plant_leaf_diseases_dataset_without_augmentation.zip  (868.0 MB)
   đã giải nén trước đó, bỏ qua


In [18]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
EXCLUDE_KW = ["__macosx"]    # nếu bộ dữ liệu có nhiều phiên bản ảnh, thêm "grayscale", "segmented" để chỉ dùng ảnh màu

def scan(root):
    rows = []
    for p in Path(root).rglob("*"):
        if (p.is_file() and p.suffix.lower() in IMG_EXT and not p.name.startswith("._")
                and not any(k in str(p).lower() for k in EXCLUDE_KW)):
            rows.append((str(p), p.parent.name))
    return pd.DataFrame(rows, columns=["path", "label"])

df = scan(RAW)
df["plant"] = df.label.str.split("___").str[0]
print("Tổng số ảnh:", len(df), "| Số lớp (theo tên thư mục):", df.label.nunique())
print("Số loài cây:", df.plant.nunique())
print(sorted(df.plant.unique()))

# Kiểm tra cấu trúc thư mục để chắc nhãn đúng
top = (df.path.str.replace(str(RAW) + "/", "", regex=False).str.split("/").str[:2].str.join("/"))
print(top.value_counts().head(15))
print("\nVí dụ nhãn:", sorted(df.label.unique())[:6])
if df.label.nunique() != 38:
    print("\n38 lớp nhưng tìm được", df.label.nunique())

Tổng số ảnh: 116934 | Số lớp (theo tên thư mục): 39
Số loài cây: 15
['Apple', 'Background_without_leaves', 'Blueberry', 'Cherry', 'Corn', 'Grape', 'Orange', 'Peach', 'Pepper,_bell', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']
path
Plant_leaf_diseases_dataset_with_augmentation/Plant_leave_diseases_dataset_with_augmentation          61486
Plant_leaf_diseases_dataset_without_augmentation/Plant_leave_diseases_dataset_without_augmentation    55448
Name: count, dtype: int64

Ví dụ nhãn: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Background_without_leaves', 'Blueberry___healthy']

38 lớp nhưng tìm được 39
